# 01 MLP Baseline

This notebook trains a simple Multi-Layer Perceptron (MLP) baseline for the sign-language image classification task.

The baseline is intentionally simple:
- grayscale images
- resize to 64 x 64
- normalization
- flattened pixel input
- one hidden layer
- fixed number of epochs
- no class weights
- no early stopping
- no data augmentation

The goal is to obtain a reference model before adding improvements.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\tobi\Documents\Machine-Learning\ps\project_submission


In [2]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

from src.dataset import load_labels, make_label_mapping, make_transforms, SignLanguageDataset, stratified_split
from src.models import SimpleMLP
from src.train_utils import get_device, train_fixed_epochs, evaluate, plot_training_curves, plot_confusion_matrix

## Data path

Change `DATA_ROOT` if your dataset is stored elsewhere.

Expected structure:
```text
sign_lang_train/sign_lang_train/
  labels.csv
  image files
```

In [3]:
DATA_ROOT = PROJECT_ROOT / "sign_lang_train" / "sign_lang_train"

df = load_labels(DATA_ROOT)
label_to_idx, idx_to_label = make_label_mapping(df["label"])

print(df.head())
print("Number of samples:", len(df))
print("Number of classes:", len(label_to_idx))
print("Classes:", list(label_to_idx.keys()))

                                          image_path label
0  c:\Users\tobi\Documents\Machine-Learning\ps\pr...     l
1  c:\Users\tobi\Documents\Machine-Learning\ps\pr...     l
2  c:\Users\tobi\Documents\Machine-Learning\ps\pr...     l
3  c:\Users\tobi\Documents\Machine-Learning\ps\pr...     l
4  c:\Users\tobi\Documents\Machine-Learning\ps\pr...     l
Number of samples: 9680
Number of classes: 36
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


## Split and dataloaders

We use a stratified train/validation/test split so that each split approximately preserves the class distribution.

In [4]:
train_df, val_df, test_df = stratified_split(df, test_size=0.2, val_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 6195
Validation: 1549
Test: 1936


In [5]:
IMAGE_SIZE = 64
BATCH_SIZE = 64

train_transform = make_transforms(image_size=IMAGE_SIZE, augment=False)
eval_transform = make_transforms(image_size=IMAGE_SIZE, augment=False)

train_dataset = SignLanguageDataset(train_df, label_to_idx=label_to_idx, transform=train_transform)
val_dataset = SignLanguageDataset(val_df, label_to_idx=label_to_idx, transform=eval_transform)
test_dataset = SignLanguageDataset(test_df, label_to_idx=label_to_idx, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Baseline training

The first baseline uses a small fixed number of epochs. This is useful to check whether a simple flattened-pixel MLP can learn the task at all.

In [6]:
device = get_device()
print("Device:", device)

model = SimpleMLP(input_size=IMAGE_SIZE * IMAGE_SIZE, hidden_size=512, num_classes=len(label_to_idx)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 8

history = train_fixed_epochs(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=NUM_EPOCHS,
)

Device: cpu
Epoch 001/8 | train_loss=2.6190 | val_loss=1.9076 | val_acc=0.4526 | val_macro_f1=0.2233
Epoch 002/8 | train_loss=1.7024 | val_loss=1.6178 | val_acc=0.5068 | val_macro_f1=0.3071
Epoch 003/8 | train_loss=1.4456 | val_loss=1.3845 | val_acc=0.5746 | val_macro_f1=0.3926
Epoch 004/8 | train_loss=1.2396 | val_loss=1.2531 | val_acc=0.6288 | val_macro_f1=0.4645
Epoch 005/8 | train_loss=1.1435 | val_loss=1.1605 | val_acc=0.6391 | val_macro_f1=0.4522
Epoch 006/8 | train_loss=1.0536 | val_loss=1.2333 | val_acc=0.6204 | val_macro_f1=0.4886
Epoch 007/8 | train_loss=0.9872 | val_loss=1.1920 | val_acc=0.6307 | val_macro_f1=0.4775
Epoch 008/8 | train_loss=0.9234 | val_loss=0.9943 | val_acc=0.6824 | val_macro_f1=0.5292


In [7]:
(PROJECT_ROOT / "saved_models").mkdir(exist_ok=True)
torch.save(model.state_dict(), PROJECT_ROOT / "saved_models" / "mlp_baseline_8epochs.pt")

plot_training_curves(
    history,
    PROJECT_ROOT / "plots" / "baseline_8epochs_training_curve.png",
    title="Baseline MLP, 8 epochs",
)

## Baseline test evaluation

Accuracy is sample-based, so frequent classes influence it more strongly. Macro-F1 first computes the F1-score for each class and then averages over all classes equally. Therefore, macro-F1 is useful for imbalanced datasets.

In [8]:
test_result = evaluate(model, test_loader, criterion, device, idx_to_label=idx_to_label)

print("Test accuracy:", test_result["accuracy"])
print("Test macro-F1:", test_result["macro_f1"])
print("Test weighted-F1:", test_result["weighted_f1"])
print(test_result["classification_report"])

plot_confusion_matrix(
    test_result["y_true"],
    test_result["y_pred"],
    class_names=[idx_to_label[i] for i in range(len(idx_to_label))],
    output_path=PROJECT_ROOT / "plots" / "baseline_8epochs_confusion_matrix.png",
    title="Baseline MLP confusion matrix",
)

Test accuracy: 0.6967975206611571
Test macro-F1: 0.5461239123542927
Test weighted-F1: 0.666749001843071
              precision    recall  f1-score   support

           0       0.76      0.86      0.81       112
           1       0.43      0.59      0.50        22
           2       0.00      0.00      0.00        22
           3       0.44      0.32      0.37        22
           4       0.75      0.69      0.72       112
           5       0.40      0.52      0.45        23
           6       0.48      0.85      0.61       112
           7       1.00      0.14      0.24        22
           8       0.80      0.12      0.21        34
           9       0.75      0.86      0.80       112
           a       0.57      0.18      0.28        22
           b       0.80      0.84      0.82        56
           c       0.87      0.93      0.90       112
           d       0.69      0.32      0.44        34
           e       0.43      0.78      0.55        23
           f       0.73      0.

## Observation

- The test accuracy of 0.70 is a good sign.
- However, the macro-F1 score is lower, which means that the model performs worse when all classes are weighted equally.
- Looking at the confusion matrix, 2 and o are not predicted at all. Most likely, o is predicted as 0, and 2 is predicted as z.
- This suggests class imbalance or weak generalization, especially for visually similar classes.

These observations motivate the improved model in the next notebook.